#  02 Homework 01 ETM

## 1. Import the needed libraries

In [163]:
from topologicpy.Vertex import Vertex
from topologicpy.Edge import Edge
from topologicpy.Face import Face
from topologicpy.Cell import Cell
from topologicpy.CellComplex import CellComplex
from topologicpy.Cluster import Cluster
from topologicpy.Topology import Topology
from topologicpy.Dictionary import Dictionary
from topologicpy.Graph import Graph
from topologicpy.Helper import Helper

## 2. Check the TopologicPy version

In [164]:
print("This tutorial requires topologicpy version 0.9.18 or newer.")
print(Helper.Version())

This tutorial requires topologicpy version 0.9.18 or newer.
The version that you are using (0.9.18) is OLDER than the latest version (0.9.21) from PyPI. Please consider upgrading to the latest version.


## 3. Set your renderer

In [165]:
renderer = "vscode"

In [166]:
import os
os.makedirs("Images", exist_ok=True)

## 4. Import th OBJs

In [167]:
OBJ_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\/R02\Rhino_Geometry_ETM.obj"

objects  = Topology.ByOBJPath(OBJ_PATH)
print(objects)

[<topologic_core.Cluster object at 0x0000020AE511E4F0>, <topologic_core.Cluster object at 0x0000020AE50BB8F0>, <topologic_core.Cluster object at 0x0000020AE4F51830>, <topologic_core.Cluster object at 0x0000020AD6F232B0>, <topologic_core.Cluster object at 0x0000020AE50BAA70>, <topologic_core.Cluster object at 0x0000020AF9D882B0>, <topologic_core.Cluster object at 0x0000020AF9D8B4B0>, <topologic_core.Cluster object at 0x0000020AF9D8B070>, <topologic_core.Cluster object at 0x0000020AF9D89270>, <topologic_core.Cluster object at 0x0000020AE4F63970>]


In [168]:
# Remove coplanar faces from each imported object and preserve original dictionaries.
cleaned_objects = []

for i, obj in enumerate(objects, start=1):
    d_obj = Topology.Dictionary(obj)
    try:
        cleaned = Topology.RemoveCoplanarFaces(
            obj,
            epsilon=0.1,
            tolerance=0.001,
            silent=True
        )
        cleaned = cleaned if cleaned else obj

        # RemoveCoplanarFaces can drop metadata; restore it for downstream name-based coloring.
        if d_obj:
            cleaned = Topology.SetDictionary(cleaned, d_obj)

        cleaned_objects.append(cleaned)
        print(f"[{i}/{len(objects)}] cleaned")
    except Exception as e:
        cleaned_objects.append(obj)
        print(f"[{i}/{len(objects)}] skipped cleaning: {e}")

objects = cleaned_objects
print(f"Cleaned objects: {len(objects)}")

[1/10] cleaned
[2/10] cleaned
[3/10] cleaned
[4/10] cleaned
[5/10] cleaned
[6/10] cleaned
[7/10] cleaned
[8/10] cleaned
[9/10] cleaned
[10/10] cleaned
Cleaned objects: 10


In [169]:
cells = []
selector = []

# Build a stable fallback name list from the original OBJ order.
raw_objects = Topology.ByOBJPath(OBJ_PATH)
if not isinstance(raw_objects, list):
    raw_objects = [raw_objects]

fallback_names = []
for robj in raw_objects:
    rd = Topology.Dictionary(robj)
    rname = str(Dictionary.ValueAtKey(rd, "name") or Dictionary.ValueAtKey(rd, "Name") or "").strip()
    fallback_names.append(rname)

for i, obj in enumerate(objects):
    d = Topology.Dictionary(obj)
    faces = Topology.Faces(obj)

    # Some operations can strip dictionaries; use original-object fallback name if needed.
    name = str(Dictionary.ValueAtKey(d, "name") or Dictionary.ValueAtKey(d, "Name") or "").strip()
    if not name and i < len(fallback_names):
        name = fallback_names[i]

    if len(faces) > 1:
        c = Cell.ByFaces(faces)

        # Fallback for non-convex or messy face sets where Cell.ByFaces can fail.
        if not c:
            cc_tmp = CellComplex.ByFaces(faces, tolerance=0.001)
            cands = Topology.Cells(cc_tmp) if cc_tmp else []
            c = cands[0] if cands else None

        if not c:
            print(f"Skipping room (cannot build cell): {name}")
            continue

        # Remove tiny mesh artifacts while preserving room solids.
        c = Topology.RemoveCollinearEdges(c)
        c_clean = Topology.RemoveCoplanarFaces(c, epsilon=0.1, tolerance=0.001, silent=True)
        c = c_clean if c_clean else c

        s = Topology.InternalVertex(c)  # Robust point guaranteed inside the cell.

        if "Entry/Kitchen" in name:
            colour = "red"
        elif "Living Room" in name:
            colour = "blue"
        elif "Bedroom" in name:
            colour = "green"
        elif "WC" in name:
            colour = "yellow"
        elif "Pantry" in name:
            colour = "cyan"
        elif "Office" in name:
            colour = "magenta"
        else:
            colour = "orange"

        d = Dictionary.SetValueAtKey(d, "colour", colour)
        d = Dictionary.SetValueAtKey(d, "vertex_size", 20)
        d = Dictionary.SetValueAtKey(d, "room_name", name)

        # Attach style dictionary to both selector and source cell.
        s = Topology.SetDictionary(s, d)
        c = Topology.SetDictionary(c, d)

        selector.append(s)
        cells.append(c)

        print(name, "->", colour)

print(len(cells))

Office -> magenta
Corridor -> orange
Bedroom1 -> green
WC1 -> yellow
WC2 -> yellow
Bedroom2 -> green
Living Room -> blue
Pantry -> cyan
Entry/Kitchen -> red
WC -> yellow
10


In [170]:
house = CellComplex.ByCells(cells)
house = Topology.TransferDictionariesBySelectors(house, selector, tranCells=True)
house_cells = Topology.Cells(house)
for house_cell in house_cells:
    d = Topology.Dictionary(house_cell)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['aabb', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[75.0, -17.0, 0.0, 99.0, 7.0, 10.0], [255, 255, 255], 'magenta', 'Office', '', 'Office', 1.0, 'Office', 20]
['aabb', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[75.0, 7.0, 0.0, 78.0, 43.0, 10.0], [255, 255, 255], 'orange', 'Corridor', '', 'Corridor', 1.0, 'Corridor', 20]
['aabb', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[78.0, 7.0, 0.0, 99.0, 22.0, 10.0], [255, 255, 255], 'green', 'Bedroom1', '', 'Bedroom1', 1.0, 'Bedroom1', 20]
['aabb', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[35.0, 0.0, 0.0, 75.0, 24.0, 10.0], [255, 255, 255], 'blue', 'Living Room', '', 'Living Room', 1.0, 'Living Room', 20]
['aabb', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[78.0, 19.0, 0.0, 86.0, 25.0, 10.0], [255, 255, 255], 'yellow', '

In [171]:
fig = Topology.Show(
    house_cells,
    selector,
    faceColorKey="colour",
    opacityKey="__none__",
    faceOpacity=0.4,
    showEdges=True,
    edgeWidth=4,
    showVertices=True,
    vertexSize=15,
    vertexLabelKey="room_name",
    showVertexLabel=True,
    renderer=renderer
)
if fig:
    fig.write_html("Images/01_rooms.html")

## 5. Graph Vertices

In [172]:
g = Graph.ByTopology(house, selector)
vert = Graph.Vertices(g)
for v in vert:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[75.0, -17.0, 0.0, 99.0, 7.0, 10.0], 0, [255, 255, 255], 'magenta', 'Office', '', 'Office', 1.0, 'Office', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[78.0, 7.0, 0.0, 99.0, 22.0, 10.0], 0, [255, 255, 255], 'green', 'Bedroom1', '', 'Bedroom1', 1.0, 'Bedroom1', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[75.0, 7.0, 0.0, 78.0, 43.0, 10.0], 0, [255, 255, 255], 'orange', 'Corridor', '', 'Corridor', 1.0, 'Corridor', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[35.0, 0.0, 0.0, 75.0, 24.0, 10.0], 0, [255, 255, 255], 'blue', 'Living Room', '', 'Living Room', 1.0, 'Living Room', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_s

In [173]:
fig = Topology.Show(g, house, vertexSizeKey="vertex_size", vertexColorKey="colour", backgroundColor="white", renderer=renderer)
if fig:
    fig.write_html("Images/02_graph_vertices.html")

## 6. Adjacency Graph

In [174]:
g_adj = Graph.ByTopology(house, direct=True, useInternalVertex=True)

adj_verts = Graph.Vertices(g_adj) or []
for v in adj_verts:
    vx, vy, vz = Vertex.X(v), Vertex.Y(v), Vertex.Z(v)

    best_s = None
    best_d = float("inf")
    for s in selector:
        sx, sy, sz = Vertex.X(s), Vertex.Y(s), Vertex.Z(s)
        d = ((vx - sx)**2 + (vy - sy)**2 + (vz - sz)**2)**0.5
        if d < best_d:
            best_d = d
            best_s = s

    sd = Topology.Dictionary(best_s) if best_s else None
    room_name = Dictionary.ValueAtKey(sd, "room_name") if sd else "room"
    color = Dictionary.ValueAtKey(sd, "colour") if sd else "black"
    size = Dictionary.ValueAtKey(sd, "vertex_size") if sd else 14

    Topology.SetDictionary(v, Dictionary.ByKeysValues(["room_name", "colour", "size"], [room_name, color, size]))

for e in Graph.Edges(g_adj) or []:
    Topology.SetDictionary(e, Dictionary.ByKeysValues(["width", "color"], [4, "black"]))

print(f"Adjacency graph — vertices: {len(Graph.Vertices(g_adj))}, edges: {len(Graph.Edges(g_adj))}")

fig = Topology.Show(
    g_adj, house,
    vertexSizeKey="size",
    vertexColorKey="colour",
    edgeWidthKey="width",
    edgeColorKey="color",
    faceColorKey="colour",
    opacityKey="__none__",
    faceOpacity=0.10,
    showEdges=True,
    backgroundColor="white",
    renderer=renderer
)
if fig:
    fig.write_html("Images/03_adjacency_graph.html")

Adjacency graph — vertices: 10, edges: 16


## 7. Adding Apertures

In [175]:
WIN_PATH  = r"C:\Users\etmaglari\IAAC\etmaglari_gML\R02\Rhino_Windows_ETM.obj"
DOOR_PATH = r"C:\Users\etmaglari\IAAC\etmaglari_gML\R02\Rhino_Doors_ETM.obj"

apertures = []

for obj in Topology.ByOBJPath(WIN_PATH):
    for face in Topology.Faces(obj):
        face = Topology.RemoveCollinearEdges(face)
        d = Dictionary.ByKeysValues(["type", "colour", "vertex_size"], ["window", "lightblue", 10])
        face = Topology.SetDictionary(face, d)
        apertures.append(face)

for obj in Topology.ByOBJPath(DOOR_PATH):
    for face in Topology.Faces(obj):
        face = Topology.RemoveCollinearEdges(face)
        d = Dictionary.ByKeysValues(["type", "colour", "vertex_size"], ["door", "brown", 10])
        face = Topology.SetDictionary(face, d)
        apertures.append(face)

print(apertures)
print(len(apertures))

[<topologic_core.Face object at 0x0000020AE4C09770>, <topologic_core.Face object at 0x0000020AE4C09DB0>, <topologic_core.Face object at 0x0000020AE4C0BFB0>, <topologic_core.Face object at 0x0000020AE4C0ADB0>, <topologic_core.Face object at 0x0000020AE4C08770>, <topologic_core.Face object at 0x0000020AE4C09670>, <topologic_core.Face object at 0x0000020AE4C0B630>, <topologic_core.Face object at 0x0000020AE4C08870>, <topologic_core.Face object at 0x0000020AE4BD3270>, <topologic_core.Face object at 0x0000020AE4C09870>, <topologic_core.Face object at 0x0000020AE4C0A2B0>, <topologic_core.Face object at 0x0000020AE4C08F70>, <topologic_core.Face object at 0x0000020AE4C09C30>, <topologic_core.Face object at 0x0000020AE4C0A970>, <topologic_core.Face object at 0x0000020AE4C0B6F0>, <topologic_core.Face object at 0x0000020AE4C0A6F0>, <topologic_core.Face object at 0x0000020AE4C0A5F0>, <topologic_core.Face object at 0x0000020AE4C0AC70>, <topologic_core.Face object at 0x0000020AE4C9D730>, <topologic_

In [176]:
house = Topology.AddApertures(house, apertures, subTopologyType="face")

In [177]:
g = Graph.ByTopology(house, direct=False, viaSharedApertures=True, toExteriorApertures=True)
verts = Graph.Vertices(g)
for v in verts:
    d = Topology.Dictionary(v)
    print(Dictionary.Keys(d), Dictionary.Values(d))

['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[75.0, -17.0, 0.0, 99.0, 7.0, 10.0], 0, [255, 255, 255], 'magenta', 'Office', '', 'Office', 1.0, 'Office', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[75.0, 7.0, 0.0, 78.0, 43.0, 10.0], 0, [255, 255, 255], 'orange', 'Corridor', '', 'Corridor', 1.0, 'Corridor', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[78.0, 7.0, 0.0, 99.0, 22.0, 10.0], 0, [255, 255, 255], 'green', 'Bedroom1', '', 'Bedroom1', 1.0, 'Bedroom1', 20]
['aabb', 'category', 'color', 'colour', 'group', 'material', 'name', 'opacity', 'room_name', 'vertex_size'] [[35.0, 0.0, 0.0, 75.0, 24.0, 10.0], 0, [255, 255, 255], 'blue', 'Living Room', '', 'Living Room', 1.0, 'Living Room', 20]
['category', 'colour', 'type', 'vertex_size'] [4, 'lightblue', 'Aperture', 10]
['category', 'colour', 

In [178]:
fig = Topology.Show(
    g, house,
    vertexSizeKey="vertex_size",
    vertexColorKey="colour",
    vertexLabelKey="room_name",
    showVertexLabel=True,
    edgeWidthKey="width",
    edgeColorKey="color",
    faceColorKey="colour",
    opacityKey="__none__",
    faceOpacity=0.10,
    showEdges=True,
    backgroundColor="white",
    renderer=renderer
)
if fig:
    fig.write_html("Images/04_aperture_graph.html")